Nicola Ranzolin 2196933
Matteo Gidoni 2209923

## Execution Environment & Data Provisioning

This notebook supports both local (Conda/Jupyter) and remote (Google Colab) execution environments. Path resolution and dataset retrieval are handled automatically upon initialization.

Execution procedure we kindly ask you to respect:

1. Initialize Environment: Run Cell 1 manually. This executes hardware detection, mounts the file system, and initiates dataset downloads.

2. Verify Data Integrity: Check the cell output. While the ETTh1 dataset will download automatically via GitHub, the electricity dataset exceeds direct-host payload limits. If the script detects missing data, it will output explicit manual download instructions. Follow them and re-run Cell 1 to validate.

Execute Pipeline: Once Cell 1 completes without missing-data warnings, proceed to execute the remainder of the notebook (Runtime -> Run All).

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch

# Notebook configuration
%matplotlib inline
plt.rcParams.update({'font.family': 'serif', 'axes.grid': True, 'grid.alpha': 0.3})

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def synchronize_device(device: torch.device) -> None:
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elif device.type == "mps":
        torch.mps.synchronize()

DEVICE = get_device()
set_seed(42)
print(f"Active device: {DEVICE}")

import urllib.request
import os

# --- Path Configuration ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = "/content/drive/MyDrive/NNDL-proj2-RANZOLIN-GIDONI"
    print("Running on Google Colab. Using Google Drive for storage.")
except ModuleNotFoundError:
    # Use the current working directory as the root if running locally
    PROJECT_ROOT = os.path.join(os.getcwd(), "NNDL-proj2")
    print(f"Running locally. Using {PROJECT_ROOT} for storage.")

# Ensure the root directory exists
os.makedirs(PROJECT_ROOT, exist_ok=True)

# Define paths relative to the PROJECT_ROOT
path_etth1 = os.path.join(PROJECT_ROOT, "data", "ETT-small", "ETTh1.csv")
path_ecl = os.path.join(PROJECT_ROOT, "data", "electricity", "electricity.csv")

# --- Dataset Downloads ---
def download_dataset(url, save_path):
    """Downloads a dataset from a URL if it doesn't already exist."""
    if not os.path.exists(save_path):
        print(f"Downloading dataset from {url}...")
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        try:
            urllib.request.urlretrieve(url, save_path)
            print(f"Successfully downloaded to {save_path}")
        except Exception as e:
            print(f"\n[!] Error downloading from {url}: {e}")
            print(f"[!] Please download the file manually and save it to: {save_path}\n")
    else:
        print(f"Dataset already exists at {save_path}. Skipping download.")

# Download ETTh1 (Small enough to be hosted directly on GitHub)
ETTH1_URL = "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv"
download_dataset(ETTH1_URL, path_etth1)

# Handle Electricity Dataset (>100MB, cannot be hosted natively on GitHub)
if not os.path.exists(path_ecl):
    print(f"\n" + "="*80)
    print(f"[ACTION REQUIRED] The electricity.csv dataset is missing!")
    print(f"Because the file is >100MB, GitHub blocks direct downloads (Returns 404).")
    print(f"1. Download it from the official Time-Series-Library Google Drive / Baidu Pan.")
    print(f"2. Unzip it and place 'electricity.csv' exactly here:")
    print(f"   -> {path_ecl}")
    print("="*80 + "\n")
else:
    print(f"Dataset already exists at {path_ecl}. Skipping download.")

Setup and dependencies

In [ ]:
# Centralized configuration registry
CONFIG_REGISTRY = {
    # Electricity Benchmark Configs
    "electricity_24": {
        "dataset_name": "electricity", "csv_path": path_ecl,
        "seq_len": 96, "pred_len": 24, "batch_size": 32, "num_workers": 0,
        "shuffle_train": True, "drop_last_train": True, "num_features": 321,
        "epochs": 20, "learning_rate": 0.001, "weight_decay": 0.0, "seed": 42,
        "fixed_period": 24, "d_model": 32, "d_ff": 64, "dropout": 0.1
    },
    "electricity_48": {
        "dataset_name": "electricity", "csv_path": path_ecl,
        "seq_len": 96, "pred_len": 48, "batch_size": 32, "num_workers": 0,
        "shuffle_train": True, "drop_last_train": True, "num_features": 321,
        "epochs": 20, "learning_rate": 0.001, "weight_decay": 0.0, "seed": 42,
        "fixed_period": 24, "d_model": 32, "d_ff": 64, "dropout": 0.1
    },
    "electricity_96": {
        "dataset_name": "electricity", "csv_path": path_ecl,
        "seq_len": 96, "pred_len": 96, "batch_size": 32, "num_workers": 0,
        "shuffle_train": True, "drop_last_train": True, "num_features": 321,
        "epochs": 20, "learning_rate": 0.001, "weight_decay": 0.0, "seed": 42,
        "fixed_period": 24, "d_model": 32, "d_ff": 64, "dropout": 0.1
    },
    # ETTh1 Benchmark Configs
    "etth1_24": {
        "dataset_name": "ETTh1", "csv_path": path_etth1,
        "seq_len": 96, "pred_len": 24, "batch_size": 32, "num_workers": 0,
        "shuffle_train": True, "drop_last_train": True, "num_features": 7,
        "epochs": 20, "learning_rate": 0.001, "weight_decay": 0.0, "seed": 42,
        "top_k": 3, "use_fft": True, "fixed_period": 24, "use_inception": True,
        "d_model": 32, "d_ff": 64, "kernel_sizes": [1, 3, 5], "dropout": 0.1
    },
    "etth1_48": {
        "dataset_name": "ETTh1", "csv_path": path_etth1,
        "seq_len": 96, "pred_len": 48, "batch_size": 32, "num_workers": 0,
        "shuffle_train": True, "drop_last_train": True, "num_features": 7,
        "epochs": 20, "learning_rate": 0.001, "weight_decay": 0.0, "seed": 42,
        "top_k": 3, "use_fft": True, "fixed_period": 24, "use_inception": True,
        "d_model": 32, "d_ff": 64, "kernel_sizes": [1, 3, 5], "dropout": 0.1
    },
    "etth1_96": {
        "dataset_name": "ETTh1", "csv_path": path_etth1,
        "seq_len": 96, "pred_len": 96, "batch_size": 32, "num_workers": 0,
        "shuffle_train": True, "drop_last_train": True, "num_features": 7,
        "epochs": 20, "learning_rate": 0.001, "weight_decay": 0.0, "seed": 42,
        "top_k": 3, "use_fft": True, "fixed_period": 24, "use_inception": True,
        "d_model": 32, "d_ff": 64, "kernel_sizes": [1, 3, 5], "dropout": 0.1
    }
}

Data Processing

In [ ]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

class TimeSeriesDataset(Dataset):
    """
    Dataset for ETTh1 and ETTm1.
    Yields input window (seq_len, num_features) and target window (pred_len, num_features).
    """
    def __init__(self, csv_path: str, flag: str, seq_len: int, pred_len: int):
        if flag not in {"train", "val", "test"}:
            raise ValueError("Flag must be train, val, or test.")
        if seq_len <= 0 or pred_len <= 0:
            raise ValueError("seq_len and pred_len must be strictly positive.")

        csv_path = Path(csv_path)
        if not csv_path.exists():
            raise FileNotFoundError(f"CSV not found: {csv_path}")

        self.seq_len = seq_len
        self.pred_len = pred_len

        df = pd.read_csv(csv_path)
        if "date" not in df.columns:
            raise ValueError("CSV must contain a 'date' column.")

        values = df.drop(columns=["date"]).to_numpy(dtype=np.float32)
        self.num_features = values.shape[1]

        n = len(values)
        train_end = int(n * 0.7)
        val_end = int(n * 0.8)

        self.scaler = StandardScaler()
        self.scaler.fit(values[:train_end])
        values = self.scaler.transform(values).astype(np.float32)
        # Interpolate localized NaNs to prevent unmasked metric collapse
        values = np.nan_to_num(values, nan=0.0)

        if flag == "train":
            split = values[:train_end]
        elif flag == "val":
            split = values[train_end - seq_len : val_end]
        else:
            split = values[val_end - seq_len :]

        self.data = torch.from_numpy(split)
        self.length = len(self.data) - seq_len - pred_len + 1

        if self.length <= 0:
            raise ValueError(f"Split too short for seq_len={seq_len}, pred_len={pred_len}")

    def __len__(self) -> int:
        return self.length

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        x_start = index
        x_end = x_start + self.seq_len
        y_start = x_end
        y_end = y_start + self.pred_len
        return self.data[x_start:x_end], self.data[y_start:y_end]

def build_dataloader(config: dict, flag: str) -> tuple[TimeSeriesDataset, DataLoader]:
    dataset = TimeSeriesDataset(
        csv_path=config["csv_path"],
        flag=flag,
        seq_len=int(config["seq_len"]),
        pred_len=int(config["pred_len"])
    )

    is_train = flag == "train"
    batch_size = int(config["batch_size"])
    num_workers = int(config.get("num_workers", 0))
    shuffle = bool(config.get("shuffle_train", True)) if is_train else False
    drop_last = bool(config.get("drop_last_train", True)) if is_train else False

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        drop_last=drop_last,
        pin_memory=bool(config.get("pin_memory", False)),
        persistent_workers=bool(config.get("persistent_workers", False)) if num_workers > 0 else False
    )
    return dataset, dataloader

Metrics & Base Neural Blocks

In [ ]:
import torch.nn as nn
from typing import Sequence

def metric_mse(pred: np.ndarray, true: np.ndarray) -> float:
    return np.mean((pred - true) ** 2)

def metric_mae(pred: np.ndarray, true: np.ndarray) -> float:
    return np.mean(np.abs(pred - true))

def metric_mase(pred: np.ndarray, true: np.ndarray) -> tuple[float, int]:
    """
    Robust Mean Absolute Scaled Error (MASE) for high-dimensional multivariate datasets.
    """
    mae_pred_per_feature = np.mean(np.abs(pred - true), axis=(0, 1))
    naive_diff = np.abs(true[:, 1:, :] - true[:, :-1, :])
    mae_naive_per_feature = np.mean(naive_diff, axis=(0, 1))

    active_sensors_mask = mae_naive_per_feature > 1e-5
    masked_count = int(np.sum(~active_sensors_mask))

    if not np.any(active_sensors_mask):
        return np.nan, masked_count

    mase_valid = mae_pred_per_feature[active_sensors_mask] / mae_naive_per_feature[active_sensors_mask]
    return float(np.mean(mase_valid)), masked_count

class InceptionBlock2D(nn.Module):
    """
    Consolidated Multi-Scale Inception Block.
    """
    def __init__(self, in_channels: int, out_channels: int, kernel_sizes: Sequence[int] = (1, 3, 5)):
        super().__init__()
        self.kernel_sizes = tuple(int(k) for k in kernel_sizes)

        self.branches = nn.ModuleList([
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=kernel,
                padding=kernel // 2,
            ) for kernel in self.kernel_sizes
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        branch_outputs = [branch(x) for branch in self.branches]
        return torch.stack(branch_outputs, dim=0).mean(dim=0)

1D Architectures (DLinear, TCN)

In [ ]:
class MovingAvg(nn.Module):
    def __init__(self, kernel_size: int, stride: int):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=stride, padding=0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        front = x[:, :, 0:1].repeat(1, 1, (self.kernel_size - 1) // 2)
        end = x[:, :, -1:].repeat(1, 1, (self.kernel_size - 1) // 2)
        return self.avg(torch.cat([front, x, end], dim=2))

class SeriesDecomp(nn.Module):
    def __init__(self, kernel_size: int = 25):
        super().__init__()
        self.moving_avg = MovingAvg(kernel_size, stride=1)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        moving_mean = self.moving_avg(x)
        return x - moving_mean, moving_mean

class DLinear(nn.Module):
    def __init__(self, seq_len: int, pred_len: int, enc_in: int):
        super().__init__()
        self.decomp = SeriesDecomp(kernel_size=25)
        self.linear_seasonal = nn.Linear(seq_len, pred_len)
        self.linear_trend = nn.Linear(seq_len, pred_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.transpose(1, 2)
        seasonal_init, trend_init = self.decomp(x)
        return (self.linear_seasonal(seasonal_init) + self.linear_trend(trend_init)).transpose(1, 2)

class Chomp1d(nn.Module):
    def __init__(self, chomp_size: int):
        super().__init__()
        self.chomp_size = chomp_size
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x[:, :, :-self.chomp_size].contiguous()

class TCNBlock(nn.Module):
    def __init__(self, n_inputs: int, n_outputs: int, kernel_size: int, stride: int, dilation: int, padding: int, dropout: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(n_inputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation),
            Chomp1d(padding), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(n_outputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation),
            Chomp1d(padding), nn.ReLU(), nn.Dropout(dropout)
        )
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class CausalTCN(nn.Module):
    def __init__(self, seq_len: int, pred_len: int, enc_in: int, num_channels: list = [32, 64], kernel_size: int = 3, dropout: float = 0.2):
        super().__init__()
        layers = []
        for i in range(len(num_channels)):
            dilation_size = 2 ** i
            in_channels = enc_in if i == 0 else num_channels[i-1]
            layers.append(TCNBlock(in_channels, num_channels[i], kernel_size, stride=1, dilation=dilation_size,
                                   padding=(kernel_size-1) * dilation_size, dropout=dropout))
        self.tcn = nn.Sequential(*layers)
        self.channel_proj = nn.Conv1d(num_channels[-1], enc_in, kernel_size=1)
        self.temporal_proj = nn.Linear(seq_len, pred_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.transpose(1, 2)
        x = self.channel_proj(self.tcn(x))
        return self.temporal_proj(x).transpose(1, 2)

2D Architectures (TimesNet & Fixed Period)

In [ ]:
import math

def fft_for_period(x: torch.Tensor, top_k: int) -> tuple[list[int], torch.Tensor]:
    batch_size, time_steps, _ = x.shape
    frequency_spectrum = torch.fft.rfft(x, dim=1)
    amplitude = frequency_spectrum.abs()

    global_amplitude = amplitude.mean(dim=(0, 2)).clone()
    if global_amplitude.numel() > 0:
        global_amplitude[0] = 0.0

    effective_top_k = min(int(top_k), max(1, global_amplitude.numel() - 1))
    _, frequency_indices = torch.topk(global_amplitude, k=effective_top_k)
    frequency_indices = frequency_indices.clamp(min=1)

    periods = [max(1, int(math.ceil(time_steps / int(idx.item())))) for idx in frequency_indices]
    period_weights = amplitude.mean(dim=2)[:, frequency_indices]

    return periods, period_weights

class TimesBlock(nn.Module):
    def __init__(self, seq_len: int, d_model: int, d_ff: int, top_k: int = 3, kernel_sizes: Sequence[int] = (1, 3, 5), dropout: float = 0.1, use_fft: bool = True, fixed_period: int = 24):
        super().__init__()
        self.seq_len, self.d_model, self.top_k, self.use_fft, self.fixed_period = seq_len, d_model, top_k, use_fft, fixed_period

        self.inception_1 = InceptionBlock2D(in_channels=d_model, out_channels=d_ff, kernel_sizes=kernel_sizes)
        self.inception_2 = InceptionBlock2D(in_channels=d_ff, out_channels=d_model, kernel_sizes=kernel_sizes)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.normalization = nn.LayerNorm(d_model)

    def _periodic_processing(self, x: torch.Tensor, period: int) -> torch.Tensor:
        batch_size, time_steps, channels = x.shape
        number_of_cycles = math.ceil(time_steps / max(1, period))
        padded_length = number_of_cycles * period

        if padded_length > time_steps:
            padding = x[:, -1:, :].repeat(1, padded_length - time_steps, 1)
            x = torch.cat([x, padding], dim=1)

        x_2d = x.reshape(batch_size, number_of_cycles, period, channels).permute(0, 3, 1, 2).contiguous()
        x_2d = self.dropout(self.activation(self.inception_1(x_2d)))
        x_2d = self.dropout(self.inception_2(x_2d))

        x_1d = x_2d.permute(0, 2, 3, 1).contiguous().reshape(batch_size, padded_length, channels)
        return x_1d[:, :time_steps, :]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        periods, period_weights = fft_for_period(x, self.top_k) if self.use_fft else ([self.fixed_period], torch.ones(x.shape[0], 1, device=x.device))

        period_outputs = [self._periodic_processing(x, period) for period in periods]
        stacked_outputs = torch.stack(period_outputs, dim=-1)
        normalized_weights = torch.softmax(period_weights, dim=1)[:, None, None, :]

        return self.normalization((stacked_outputs * normalized_weights).sum(dim=-1) + x)

class TimesNetOriginal(nn.Module):
    def __init__(self, seq_len: int, pred_len: int, enc_in: int, d_model: int = 32, d_ff: int = 64, top_k: int = 3, num_blocks: int = 2, fixed_period: int = 24, dropout: float = 0.1):
        super().__init__()
        self.embedding = nn.Linear(enc_in, d_model)
        self.times_blocks = nn.ModuleList([
            TimesBlock(seq_len, d_model, d_ff, top_k, use_fft=True, fixed_period=fixed_period, dropout=dropout)
            for _ in range(num_blocks)
        ])
        self.temporal_projection = nn.Linear(seq_len, pred_len)
        self.dropout = nn.Dropout(dropout)
        self.output_projection = nn.Linear(d_model, enc_in)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.embedding(x)
        for block in self.times_blocks:
            features = block(features)
        forecast = self.temporal_projection(features.transpose(1, 2))
        forecast = self.dropout(forecast).transpose(1, 2)
        return self.output_projection(forecast)

class FixedPeriodInception2D(TimesNetOriginal):
    """Reuses TimesNetOriginal logic but explicitly disables FFT"""
    def __init__(self, seq_len: int, pred_len: int, num_features: int, period: int = 24, d_model: int = 32, d_ff: int = 64, num_blocks: int = 1, dropout: float = 0.1):
        super().__init__(seq_len, pred_len, num_features, d_model, d_ff, top_k=1, num_blocks=num_blocks, fixed_period=period, dropout=dropout)
        self.times_blocks = nn.ModuleList([
            TimesBlock(seq_len, d_model, d_ff, top_k=1, use_fft=False, fixed_period=period, dropout=dropout)
            for _ in range(num_blocks)
        ])

Architectural Efficiency Backbones

In [ ]:
class DepthwiseSeparableBlock2D(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1, groups=in_channels),
            nn.Conv2d(in_channels, out_channels, kernel_size=1),
            nn.GELU()
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class GroupConvBlock2D(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, groups: int = 4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, groups=groups),
            nn.GELU(), nn.Conv2d(out_channels, out_channels, kernel_size=1), nn.GELU()
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class LightTimesBlock(nn.Module):
    def __init__(self, d_model: int, fixed_period: int, block_type: str, kernel_sizes=(1, 3, 5), groups=4):
        super().__init__()
        self.fixed_period = max(1, int(fixed_period))
        if block_type == "multiscale": self.conv_2d = InceptionBlock2D(d_model, d_model, kernel_sizes)
        elif block_type == "depthwise": self.conv_2d = DepthwiseSeparableBlock2D(d_model, d_model)
        elif block_type == "group": self.conv_2d = GroupConvBlock2D(d_model, d_model, groups)
        else: self.conv_2d = nn.Sequential(nn.Conv2d(d_model, d_model, 3, padding=1), nn.GELU())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        cycles = math.ceil(T / self.fixed_period)
        L = cycles * self.fixed_period

        x_pad = torch.cat([x, x[:, -1:, :].repeat(1, L - T, 1)], dim=1) if L > T else x
        x_2d = x_pad.reshape(B, cycles, self.fixed_period, C).permute(0, 3, 1, 2).contiguous()
        return self.conv_2d(x_2d).permute(0, 2, 3, 1).reshape(B, L, C)[:, :T, :] + x

class LightTimesNet(nn.Module):
    def __init__(self, seq_len: int, pred_len: int, enc_in: int, d_model: int, fixed_period: int, block_type: str, num_blocks: int):
        super().__init__()
        self.embedding = nn.Linear(enc_in, d_model)
        self.blocks = nn.ModuleList([LightTimesBlock(d_model, fixed_period, block_type) for _ in range(num_blocks)])
        self.proj, self.out = nn.Linear(seq_len, pred_len), nn.Linear(d_model, enc_in)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        encoded = self.embedding(x)
        for b in self.blocks: encoded = b(encoded)
        return self.out(self.proj(encoded.transpose(1, 2)).transpose(1, 2))

Training Engine (Refactored for in-memory execution)

In [ ]:
import time
from torch import optim


class EarlyStopping:
    """
    Tracks validation loss and saves the best model state in memory.
    Triggers early stopping if validation loss does not improve for `patience` epochs.
    """
    def __init__(self, patience: int = 5):
        self.patience = patience
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_state = None
        self.best_epoch_idx = 0

    def __call__(self, val_loss: float, model: nn.Module, epoch_idx: int) -> None:
        score = -val_loss
        if self.best_score is None or score > self.best_score:
            self.best_score = score
            self.best_epoch_idx = epoch_idx
            self.best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

def execute_training_run(config_name: str, model_name: str, **overrides):
    """
    Executes model instantiation, optimization, and evaluation strictly
    driven by the unified configuration registry.
    """
    if config_name not in CONFIG_REGISTRY:
        raise KeyError(f"Configuration '{config_name}' not defined in CONFIG_REGISTRY.")

    cfg = CONFIG_REGISTRY[config_name].copy()
    cfg.update(overrides)

    set_seed(cfg.get("seed", 42))

    num_features = cfg["num_features"]
    seq_len = cfg["seq_len"]
    pred_len = cfg["pred_len"]
    d_model = cfg.get("d_model", 32)
    d_ff = cfg.get("d_ff", 64)
    dropout = cfg.get("dropout", 0.1)

    period = cfg.get("fixed_period", 24)
    top_k = cfg.get("top_k", 3)
    num_blocks = cfg.get("num_blocks", 1)

    models_dict = {
        "DLinear": lambda: DLinear(
            seq_len=seq_len, pred_len=pred_len, enc_in=num_features
        ),
        "CausalTCN": lambda: CausalTCN(
            seq_len=seq_len, pred_len=pred_len, enc_in=num_features, dropout=dropout
        ),
        "TimesNetOriginal": lambda: TimesNetOriginal(
            seq_len=seq_len, pred_len=pred_len, enc_in=num_features,
            d_model=d_model, d_ff=d_ff, top_k=top_k, num_blocks=num_blocks,
            dropout=dropout, fixed_period=period
        ),
        "FixedPeriodInception": lambda: FixedPeriodInception2D(
            seq_len=seq_len, pred_len=pred_len, num_features=num_features,
            period=period, d_model=d_model, d_ff=d_ff, num_blocks=num_blocks,
            dropout=dropout
        ),
        "LightTimesNet_MultiScale": lambda: LightTimesNet(
            seq_len=seq_len, pred_len=pred_len, enc_in=num_features,
            d_model=d_model, fixed_period=period, block_type="multiscale", num_blocks=num_blocks
        ),
        "LightTimesNet_Depthwise": lambda: LightTimesNet(
            seq_len=seq_len, pred_len=pred_len, enc_in=num_features,
            d_model=d_model, fixed_period=period, block_type="depthwise", num_blocks=num_blocks
        ),
        "LightTimesNet_Group": lambda: LightTimesNet(
            seq_len=seq_len, pred_len=pred_len, enc_in=num_features,
            d_model=d_model, fixed_period=period, block_type="group", num_blocks=num_blocks
        ),
        "LightTimesNet_SingleKernel": lambda: LightTimesNet(
            seq_len=seq_len, pred_len=pred_len, enc_in=num_features,
            d_model=d_model, fixed_period=period, block_type="single_kernel", num_blocks=num_blocks
        ),
    }

    if model_name not in models_dict:
        raise ValueError(f"Model '{model_name}' not implemented in factory.")

    model = models_dict[model_name]().to(DEVICE)

    _, train_loader = build_dataloader(cfg, "train")
    _, val_loader = build_dataloader(cfg, "val")
    _, test_loader = build_dataloader(cfg, "test")

    optimizer = optim.Adam(
        model.parameters(),
        lr=cfg["learning_rate"],
        weight_decay=cfg.get("weight_decay", 0.0)
    )
    criterion = nn.MSELoss()
    early_stopping = EarlyStopping(patience=cfg.get("patience", 5))

    # --- Training Loop with History Tracking ---
    start_time = time.perf_counter()
    train_history = {"train_mse": [], "val_mse": [], "epoch_time_seconds": []}

    for epoch in range(cfg["epochs"]):
        model.train()
        train_loss = []
        epoch_start = time.perf_counter()

        for batch_x, batch_y in train_loader:
            optimizer.zero_grad(set_to_none=True)
            out = model(batch_x.to(DEVICE))
            loss = criterion(out, batch_y.to(DEVICE))
            train_loss.append(loss.item() * batch_x.size(0))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

        epoch_time = time.perf_counter() - epoch_start
        train_mse_epoch = np.sum(train_loss) / len(train_loader.dataset)

        model.eval()
        val_loss = []
        with torch.no_grad():
            for bx, by in val_loader:
                out = model(bx.to(DEVICE))
                val_loss.append(criterion(out, by.to(DEVICE)).item() * bx.size(0))

        val_mse_epoch = np.sum(val_loss) / len(val_loader.dataset)

        train_history["train_mse"].append(float(train_mse_epoch))
        train_history["val_mse"].append(float(val_mse_epoch))
        train_history["epoch_time_seconds"].append(float(epoch_time))

        early_stopping(val_mse_epoch, model, epoch)
        if early_stopping.early_stop:
            break

    total_training_time = time.perf_counter() - start_time

    # --- Evaluation ---
    model.load_state_dict(early_stopping.best_state)
    model.eval()

    preds, trues = [], []
    with torch.no_grad():
        for bx, by in test_loader:
            preds.append(model(bx.to(DEVICE)).cpu().numpy())
            trues.append(by.numpy())

    preds = np.concatenate(preds, axis=0)
    trues = np.concatenate(trues, axis=0)

    mse = metric_mse(preds, trues)
    mae = metric_mae(preds, trues)
    mase_val, masked_count = metric_mase(preds, trues)

    metrics = {
        "model": model_name,
        "config": config_name,
        "dataset": cfg["dataset_name"],
        "num_features": num_features,
        "seq_len": seq_len,
        "pred_len": pred_len,
        "test_mse": float(mse),
        "test_mae": float(mae),
        "test_mase": float(mase_val),
        "masked_features": masked_count,
        "training_time_s": float(total_training_time),
        "params": sum(p.numel() for p in model.parameters() if p.requires_grad),
        "best_epoch_idx": early_stopping.best_epoch_idx
    }

    return metrics, train_history, early_stopping.best_state

Execution & Logging

In [ ]:
# =============================================================================
# ABLATION PIPELINE EXECUTION
# =============================================================================
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

configs = [
    "etth1_24", "etth1_48", "etth1_96",
    "electricity_24", "electricity_48", "electricity_96"
]

unique_commands = set()

# GROUP 1: Sequence Length Benchmark
for config in configs:
    for seq in [96, 192, 384]:
        unique_commands.add((config, "DLinear", tuple({"seq_len": seq}.items())))
        unique_commands.add((config, "CausalTCN", tuple({"seq_len": seq}.items())))
        unique_commands.add((config, "FixedPeriodInception", tuple({"seq_len": seq, "fixed_period": 24, "num_blocks": 1}.items())))
    unique_commands.add((config, "TimesNetOriginal", tuple({"seq_len": 96, "top_k": 2, "num_blocks": 1}.items())))

# GROUP 2: TimesNet Ablation
for config in configs:
    for k in [1, 2, 3]:
        for b in [1, 2, 3]:
            unique_commands.add((config, "TimesNetOriginal", tuple({"seq_len": 96, "top_k": k, "num_blocks": b}.items())))

# GROUP 3: FixedPeriod Ablation
for config in configs:
    for p in [17, 24, 48]:
        for b in [1, 2, 3]:
            unique_commands.add((config, "FixedPeriodInception", tuple({"seq_len": 96, "fixed_period": p, "num_blocks": b}.items())))

# GROUP 4: Spatial Backbone Ablation
backbones = [
    "LightTimesNet_MultiScale",
    "LightTimesNet_Depthwise",
    "LightTimesNet_Group",
    "LightTimesNet_SingleKernel"
]
for config in configs:
    for bb in backbones:
        unique_commands.add((config, bb, tuple({"seq_len": 96, "fixed_period": 24, "num_blocks": 1}.items())))

experiment_suite = [
    (cfg, mod, dict(ovr))
    for cfg, mod, ovr in sorted(list(unique_commands), key=lambda x: (x[0], x[1]))
]

print(f"Total unique experiments scheduled: {len(experiment_suite)}")

results_registry = []
history_registry = {}

# Safeguard: Initialize empty DataFrame before loop to prevent NameError on interruption
df_results = pd.DataFrame()

# Base directory for storing individual experiment folders
experiments_root = os.path.join(PROJECT_ROOT, "experiments")
os.makedirs(experiments_root, exist_ok=True)
output_csv = os.path.join(PROJECT_ROOT, "full_ablation_results.csv")

for idx, (config_key, model_key, overrides) in enumerate(experiment_suite):
    print(f"\n[{idx+1}/{len(experiment_suite)}] Executing: {model_key} | {config_key} | {overrides}")
    try:
        metrics, history, best_state = execute_training_run(config_key, model_key, **overrides)

        # Retroactive metadata injection
        metrics['fixed_period'] = overrides.get('fixed_period', CONFIG_REGISTRY[config_key].get('fixed_period', None))
        metrics['top_k'] = overrides.get('top_k', CONFIG_REGISTRY[config_key].get('top_k', None))
        metrics['num_blocks'] = overrides.get('num_blocks', CONFIG_REGISTRY[config_key].get('num_blocks', 1))

        results_registry.append(metrics)

        # Create dedicated experiment folder and Naming Convention
        dataset_name = CONFIG_REGISTRY[config_key]['dataset_name']
        exp_name = f"{model_key}_{dataset_name}_S{metrics['seq_len']}_H{metrics['pred_len']}"
        if model_key == "TimesNetOriginal":
            exp_name += f"_K{metrics['top_k']}_B{metrics['num_blocks']}"
        elif model_key not in ["DLinear", "CausalTCN"]:
            exp_name += f"_P{metrics['fixed_period']}_B{metrics['num_blocks']}"

        exp_dir = os.path.join(experiments_root, exp_name)
        os.makedirs(exp_dir, exist_ok=True)
        history_registry[exp_name] = {"history": history, "best_epoch_idx": metrics["best_epoch_idx"]}

        # 1. Save Metrics JSON
        with open(os.path.join(exp_dir, "metrics.json"), "w") as f:
            json.dump(metrics, f, indent=4)

        # 2. Save History JSON
        with open(os.path.join(exp_dir, "history.json"), "w") as f:
            json.dump(history, f, indent=4)

        # 3. Save Model Weights (best_model.pth)
        torch.save(best_state, os.path.join(exp_dir, "best_model.pth"))

        # 4. Save Learning Curves PDF using custom function
        def save_learning_curves_inline(hist, best_idx, save_path):
            train_loss = hist["train_mse"]
            val_loss = hist["val_mse"]
            epochs = np.arange(1, len(train_loss) + 1)
            best_ep = epochs[best_idx]
            best_v = val_loss[best_idx]
            fig, ax = plt.subplots(figsize=(10, 6))
            ax.plot(epochs, train_loss, color='blue', label='Train. error', linewidth=2)
            ax.plot(epochs, val_loss, color='red', label='Valid. error', linewidth=2)
            ok_start = max(1, best_ep - 1)
            ok_end = min(len(epochs), best_ep + 1)
            y_max, y_min = max(max(train_loss), max(val_loss)), min(min(train_loss), min(val_loss))
            text_y = y_min + (y_max - y_min) * 0.85
            ax.axvspan(1, ok_start, facecolor='#E8D8D8', alpha=0.6)
            if ok_start > 1: ax.text((1 + ok_start)/2, text_y, 'UNDERFITTING', ha='center', va='center', fontweight='bold')
            ax.axvspan(ok_start, ok_end, facecolor='#D8E8D8', alpha=0.6)
            ax.text(best_ep, text_y, 'OK', ha='center', va='center', fontweight='bold')
            if ok_end < len(epochs):
                ax.axvspan(ok_end, len(epochs), facecolor='#F8D8D8', alpha=0.6)
                ax.text((ok_end + len(epochs))/2, text_y, 'OVERFITTING', ha='center', va='center', fontweight='bold')
            ax.annotate(r'STOP HERE' + '\n' + r'$\mathcal{W}^*$', xy=(best_ep, best_v), xytext=(best_ep, best_v + (y_max - y_min)*0.15),
                        arrowprops=dict(facecolor='black', shrink=0.05, width=2, headwidth=8), ha='center', va='bottom', fontsize=14, fontweight='bold')
            ax.set_xlabel('Training epochs', fontweight='bold', loc='right')
            ax.set_ylabel(r'$\mathcal{L}$', fontsize=20, rotation=0, labelpad=15, loc='top')
            ax.set_title('Training-validation curves', fontsize=22, loc='left', color='#003300')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['bottom'].set_linewidth(2)
            ax.spines['left'].set_linewidth(2)
            ax.legend(frameon=False, loc='upper right')
            plt.tight_layout()
            plt.savefig(save_path, format='pdf', dpi=300)
            plt.close(fig)

        save_learning_curves_inline(history, metrics["best_epoch_idx"], os.path.join(exp_dir, "learning_curves.pdf"))

        print(f" -> MSE: {metrics['test_mse']:.4f} | MAE: {metrics['test_mae']:.4f} | Params: {metrics['params']}")

        # Intermediate disk saving to prevent data loss on Colab timeout
        df_results = pd.DataFrame(results_registry)
        df_results['trainable_parameters'] = df_results['params']
        df_results.to_csv(output_csv, index=False)

    except Exception as e:
        print(f" -> Execution Failed: {e}")

if not df_results.empty:
    display(df_results[['config', 'dataset', 'model', 'seq_len', 'pred_len', 'test_mse', 'test_mae', 'test_mase', 'params']])
else:
    print("No experiments completed. DataFrame is empty.")

plotting results

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset

def display_pivot_tables(df: pd.DataFrame):
    if df.empty:
        print("Dataframe is empty. Skipping pivot tables.")
        return

    print("=" * 80)
    print("  REPORT TABLES GENERATION")
    print("=" * 80)

    # --- TABLE 1: Cycles ---
    cond_1d = df['model'].isin(['DLinear', 'CausalTCN'])
    cond_fp = (df['model'] == 'FixedPeriodInception') & (df['fixed_period'] == 24) & (df['num_blocks'] == 1)
    pt1 = pd.pivot_table(df[cond_1d | cond_fp], values=['test_mae', 'test_mse'],
                         index=['dataset', 'pred_len'], columns=['seq_len', 'model'], aggfunc='first')

    if not pt1.empty:
        avg1 = pt1.groupby(level='dataset').mean()
        avg1.index = pd.MultiIndex.from_product([avg1.index, ['Avg']])
        pt1 = pd.concat([pt1, avg1]).sort_index(level=0)
        print("\nTable 1: Context Scalability (Cicli)")
        display(pt1)

    # --- TABLE 2: Period Sensitivity ---
    t2_df = df[(df['model'] == 'FixedPeriodInception') & (df['seq_len'] == 96) & (df['num_blocks'] == 1)]
    pt2 = pd.pivot_table(t2_df, values=['test_mae', 'test_mse'],
                         index=['dataset', 'pred_len'], columns=['fixed_period'], aggfunc='first')

    if not pt2.empty:
        dl_df = df[(df['model'] == 'DLinear') & (df['seq_len'] == 96)]
        if not dl_df.empty:
            dl_mse = dl_df.set_index(['dataset', 'pred_len'])['test_mse'].to_dict()
            dl_mae = dl_df.set_index(['dataset', 'pred_len'])['test_mae'].to_dict()
            pt2[('test_mse', 'DLinear')] = [dl_mse.get(idx, np.nan) for idx in pt2.index]
            pt2[('test_mae', 'DLinear')] = [dl_mae.get(idx, np.nan) for idx in pt2.index]

        avg2 = pt2.groupby(level='dataset').mean()
        avg2.index = pd.MultiIndex.from_product([avg2.index, ['Avg']])
        pt2 = pd.concat([pt2, avg2]).sort_index(level=0)
        print("\nTable 2: Period Sensitivity")
        display(pt2)

    # --- TABLE 3: Times Block vs Frequency ---
    t3_tn = df[(df['model'] == 'TimesNetOriginal') & (df['seq_len'] == 96)].copy()
    t3_fp = df[(df['model'] == 'FixedPeriodInception') & (df['seq_len'] == 96) & (df['fixed_period'] == 24)].copy()
    t3_fp['top_k'] = 'N/A'
    pt3 = pd.pivot_table(pd.concat([t3_tn, t3_fp]), values=['test_mae', 'test_mse'],
                         index=['dataset', 'model', 'top_k'], columns=['pred_len', 'num_blocks'], aggfunc='first')

    if not pt3.empty:
        print("\nTable 3: Times Block vs Frequency (Top-K)")
        display(pt3)

    # --- TABLE 4: Backbone Efficiency ---
    bb_models = ['LightTimesNet_MultiScale', 'LightTimesNet_Depthwise', 'LightTimesNet_Group', 'LightTimesNet_SingleKernel']
    cond_tn = (df['model'] == 'TimesNetOriginal') & (df['top_k'] == 2) & (df['num_blocks'] == 1)
    t4_df = df[(df['seq_len'] == 96) & (cond_tn | df['model'].isin(bb_models) | (df['model'] == 'DLinear'))]
    pt4 = pd.pivot_table(t4_df, values=['test_mse', 'test_mae', 'test_mase', 'trainable_parameters'],
                         index=['dataset', 'model', 'pred_len'], aggfunc='first')

    if not pt4.empty:
        print("\nTable 4: Spatial Backbone Efficiency")
        display(pt4)

def plot_experiments_inline(df: pd.DataFrame, out_dir: str, dataset: str = "ETTh1", pred_len: int = 96):
    df_base = df[(df['dataset'] == dataset) & (df['pred_len'] == pred_len)].copy()
    if df_base.empty: return

    plt.rcParams.update({'font.family': 'serif', 'axes.grid': True, 'grid.alpha': 0.3})
    base_path = os.path.join(out_dir, f"ablation_analysis_{dataset}_H{pred_len}")

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f"Ablation Analysis: {dataset.upper()} (H={pred_len})", fontsize=16, fontweight='bold')

    # --- PLOT A: Temporal Context Scalability ---
    ax_a = axes[0, 0]
    m1d = df_base[df_base['model'].isin(['DLinear', 'CausalTCN'])]
    m2d = df_base[(df_base['model'] == 'FixedPeriodInception') & (df_base['fixed_period'] == 24) & (df_base['num_blocks'] == 1)]
    plot_df = pd.concat([m1d, m2d]).sort_values('seq_len')

    for model in ['DLinear', 'CausalTCN', 'FixedPeriodInception']:
        subset = plot_df[plot_df['model'] == model]
        if not subset.empty:
            ax_a.plot(subset['seq_len'], subset['test_mse'], marker='o', linewidth=2, label=model)

    ax_a.set_title('A. Temporal Context Scalability (seq_len)', fontweight='bold')
    ax_a.set_xlabel('Sequence Length')
    ax_a.set_ylabel('Test MSE')
    ax_a.set_xticks([96, 192, 384])
    ax_a.legend()

    # --- PLOT B: FFT Extraction Cost vs Fixed Baseline ---
    ax_b = axes[0, 1]
    tn_df = df_base[(df_base['model'] == 'TimesNetOriginal') & (df_base['seq_len'] == 96)].sort_values('top_k')
    fp_df = df_base[(df_base['model'] == 'FixedPeriodInception') & (df_base['seq_len'] == 96) & (df_base['fixed_period'] == 24)]
    colors = plt.cm.tab10.colors

    for i, b in enumerate([1, 2, 3]):
        color = colors[i % len(colors)]
        subset_tn = tn_df[tn_df['num_blocks'] == b]
        if not subset_tn.empty:
            ax_b.plot(subset_tn['top_k'], subset_tn['test_mse'], marker='s', linewidth=2, color=color, label=f'TimesNet (B={b})')
        subset_fp = fp_df[fp_df['num_blocks'] == b]
        if not subset_fp.empty:
            ax_b.axhline(y=subset_fp['test_mse'].values[0], linestyle='--', linewidth=2, color=color, label=f'FixedPeriod (B={b})')

    ax_b.set_title('B. FFT Extraction Cost vs Fixed Baseline', fontweight='bold')
    ax_b.set_xlabel('Top K Frequencies')
    ax_b.set_ylabel('Test MSE')
    if not tn_df.empty: ax_b.set_xticks(sorted(tn_df['top_k'].dropna().unique()))
    ax_b.legend(ncol=2)

    # --- PLOT C: Domain Knowledge Injection ---
    ax_c = axes[1, 0]
    fp_df_c = df_base[(df_base['model'] == 'FixedPeriodInception') & (df_base['seq_len'] == 96) & (df_base['num_blocks'] == 1)].sort_values('fixed_period')
    if not fp_df_c.empty:
        bars = ax_c.bar([str(int(p)) for p in fp_df_c['fixed_period'].dropna()], fp_df_c['test_mse'], color='#4C72B0', edgecolor='black')
        ax_c.set_title('C. Domain Knowledge Injection', fontweight='bold')
        ax_c.set_xlabel('Fixed Period')
        ax_c.set_ylabel('Test MSE')
        ax_c.set_ylim(0, fp_df_c['test_mse'].max() * 1.2)
        for bar in bars:
            ax_c.text(bar.get_x() + bar.get_width()/2, bar.get_height() + (fp_df_c['test_mse'].max() * 0.02),
                      f'{bar.get_height():.3f}', ha='center', fontweight='bold')

    # --- PLOT D: Spatial Backbone Efficiency (Pareto Front) ---
    ax_d = axes[1, 1]
    bb_models = ['LightTimesNet_MultiScale', 'LightTimesNet_Depthwise', 'LightTimesNet_Group', 'LightTimesNet_SingleKernel', 'DLinear']
    bb_df = df_base[(df_base['model'].isin(bb_models)) & (df_base['seq_len'] == 96)].copy()

    if not bb_df.empty:
        markers = ['*', 'o', 's', 'X', 'D']
        pareto_points = []

        for _, row in bb_df.iterrows():
            if pd.isna(row.get('trainable_parameters')): continue
            x_val, y_val = row['trainable_parameters'], row['test_mse']
            marker = markers[bb_models.index(row['model'])]
            model_clean = row['model'].replace('LightTimesNet_', '')
            ax_d.scatter(x_val, y_val, label=model_clean, marker=marker, s=200, edgecolors='black', zorder=3)
            pareto_points.append((x_val, y_val))

        pareto_points.sort(key=lambda p: (p[0], p[1]))
        front, min_y = [], float('inf')
        for x, y in pareto_points:
            if y < min_y:
                front.append((x, y))
                min_y = y

        if len(front) > 1:
            ax_d.plot([p[0] for p in front], [p[1] for p in front], 'k--', alpha=0.6, linewidth=2, label='Pareto Front', zorder=2)

        ax_d.set_title('D. Spatial Backbone Efficiency', fontweight='bold')
        ax_d.set_xlabel('Trainable Parameters')
        ax_d.set_ylabel('Test MSE')
        ax_d.set_xscale('log')
        ax_d.legend()

    plt.tight_layout()
    plt.savefig(f"{base_path}_dashboard.png", dpi=300, bbox_inches='tight')
    plt.show()

# =============================================================================
# FINAL EXECUTION
# =============================================================================
display_pivot_tables(df_results)

for h in [24, 48, 96]:
    plot_experiments_inline(df_results, out_dir=PROJECT_ROOT, dataset="ETTh1", pred_len=h)
    plot_experiments_inline(df_results, out_dir=PROJECT_ROOT, dataset="electricity", pred_len=h)